# 112 — CPTAC-3 RNA-seq Technical Characterization and Source QC Provenance

## Scientific objective

Characterize the audited primary-tumor STAR Counts source files from GDC CPTAC-3 and assess the availability and provenance of sequencing-quality metadata without assigning sample-level QC eligibility.

The notebook consumes source identities and acquisition-integrity evidence generated by **Notebook 111**, together with two version-controlled GDC source metadata snapshots under `docs/evidence/phase1/notebook112/`. It does not read outputs from Notebook 113 or any Phase 8 notebook.

The source-only analysis identifies uniquely represented primary RNA-seq cases, reads their local STAR `unstranded` count files, examines additional same-case RNA files, and reconstructs a bounded, deterministic 12-file source-metadata audit. It distinguishes metadata agreement from independent sequencing-quality certification.

## Downstream outputs

Exactly two generated files are persisted under `data/interim/metadata/`:

- `112_cptac3_rna_simple_primary_technical_inventory.csv`: file- and case-keyed technical inventory for 1,082 uniquely represented primary candidates.
- `112_cptac3_qc_provenance_handoff.json`: SHA-256-fingerprinted source provenance, output identity, descriptive audit results and unresolved QC limitations.

Both outputs are consumed or authenticated by Notebook 113. Intermediate descriptive tables remain visible in notebook outputs; they are not treated as independently frozen artifacts.

## Interpretation

File integrity and raw-count validity do **not** constitute independent read-level/library QC. No expression normalization, program scoring, TCGA lineage assignment, clinical prediction, sample-level QC exclusion, or donor-overlap inference is performed here. Frozen program-gene compatibility and proposed cohort construction belong to Notebook 113; lineage-aware inferential replication belongs to Notebook 800.

Execution order: **111 → 112 → 113 → 800**.


In [ ]:
# =============================================================================
# Imports, authoritative 111 handoff, and immutable source evidence
# =============================================================================
import hashlib
import json
import math
import platform

import numpy as np
import pandas as pd
from IPython.display import display

from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths, project_relative_path

ROOT = Paths.root
META = Paths.metadata
EVIDENCE = ROOT / "docs/evidence/phase1/notebook112"
AUDIT_111_PATH = META / "111_cptac3_acquisition_audit_summary.json"
FILE_INDEX_111_PATH = META / "111_cptac3_file_index.csv"
PRIMARY_111_PATH = META / "111_cptac3_provisional_primary_tumor_files.csv"
STAR_GDC_SNAPSHOT_PATH = EVIDENCE / "112_gdc_star_output_pilot_snapshot.json"
SNAPSHOT_112_PATH = EVIDENCE / "112_gdc_upstream_input_pilot_snapshot.json"
STAR_GDC_SNAPSHOT_SHA256 = (
    "dcd1dc8b332bda55c658751e6e781a9cbe3719ed07fd17e38e67bbcf794b6ecb"
)
SCAN_OUTPUT = META / "112_cptac3_rna_simple_primary_technical_inventory.csv"

OUTPUT_112_PATHS = {
    "handoff": META / "112_cptac3_qc_provenance_handoff.json",
}
QC_FIELDS = (
    "total_reads", "proportion_reads_mapped",
    "proportion_reads_duplicated", "average_base_quality",
    "average_read_length", "average_insert_size",
)
for path in (
    AUDIT_111_PATH,
    FILE_INDEX_111_PATH,
    PRIMARY_111_PATH,
    STAR_GDC_SNAPSHOT_PATH,
):
    if not path.is_file():
        raise FileNotFoundError("Missing required source evidence: " + str(path))
if calculate_sha256(STAR_GDC_SNAPSHOT_PATH) != STAR_GDC_SNAPSHOT_SHA256:
    raise ValueError(
        "Version-controlled STAR-output source snapshot fingerprint mismatch"
    )
with AUDIT_111_PATH.open(encoding="utf-8") as handle:
    audit_111 = json.load(handle)
if audit_111.get("full_payload_md5_verified") is not True:
    raise ValueError("Notebook 111 full source-payload MD5 verification is required")
file_info = audit_111["audit_artifacts"]["file_index"]
if calculate_sha256(FILE_INDEX_111_PATH) != file_info["sha256"]:
    raise ValueError("Notebook 111 source file-index hash mismatch")
index_111 = pd.read_csv(FILE_INDEX_111_PATH, dtype="string")
primary_111 = pd.read_csv(PRIMARY_111_PATH, dtype="string")
idx_rna = index_111.loc[index_111["modality"].eq("rnaseq")].copy()
if len(idx_rna) != 2968:
    raise ValueError("Notebook 111 RNA source universe changed")
print("Audited source RNA files:", len(idx_rna))
print(
    "Versioned STAR-output source snapshot:",
    project_relative_path(STAR_GDC_SNAPSHOT_PATH),
)


## Primary-Tumor RNA-seq Source Candidates

Use only the independently audited sample, aliquot and file associations from Notebook 111. Retain cases with exactly one provisionally primary RNA-seq file, sample and aliquot as a descriptive technical inventory. Multiple primary records are accounted for separately rather than resolved by arbitrary file selection.


In [ ]:
# =============================================================================
# Select uniquely represented primary RNA-seq source cases
# =============================================================================
primary_rna = primary_111.loc[primary_111["modality"].eq("rnaseq")].copy()
necessary = {"file_id", "case_uuid", "sample_uuid", "aliquot_uuid",
             "filename", "size_bytes", "provider_md5", "primary_site", "disease_type"}
if not necessary.issubset(primary_rna.columns):
    raise KeyError(
        f"Missing primary-source columns: {necessary - set(primary_rna.columns)}"
    )
case_structure = primary_rna.groupby("case_uuid").agg(
    n_files=("file_id", "nunique"),
    n_samples=("sample_uuid", "nunique"),
    n_aliquots=("aliquot_uuid", "nunique"),
)
simple_case_ids = case_structure.loc[
    case_structure[["n_files", "n_samples", "n_aliquots"]].eq(1).all(axis=1)
].index
rna_e_simple_files = (
    primary_rna.loc[primary_rna["case_uuid"].isin(simple_case_ids)]
    .sort_values(["case_uuid", "file_id"], kind="stable")
    .reset_index(drop=True)
)
if (len(primary_rna) != 1731 or case_structure.shape[0] != 1370
        or len(simple_case_ids) != 1082 or len(rna_e_simple_files) != 1082
        or rna_e_simple_files["case_uuid"].duplicated().any()
        or rna_e_simple_files["file_id"].duplicated().any()):
    raise ValueError(
        "Primary source candidate classification differs from audited inputs"
    )
rna_all_source = idx_rna.copy()
source_counts = rna_all_source.groupby("case_uuid").agg(
    n_all_rna_files=("file_id", "nunique"),
    n_ambiguous_rna_files=("mapping_status",
        lambda v: int(v.eq("ambiguous_sample_ancestry").sum())),
)
rna_e_simple_files["n_other_rna_files_in_case"] = (
    rna_e_simple_files["case_uuid"]
    .map(source_counts["n_all_rna_files"])
    .astype(int)
    - 1
)
rna_e_simple_files["n_unresolved_ancestry_rna_files"] = (
    rna_e_simple_files["case_uuid"]
    .map(source_counts["n_ambiguous_rna_files"])
    .astype(int)
)
if (int(rna_e_simple_files["n_other_rna_files_in_case"].gt(0).sum()) != 530
        or int(
            rna_e_simple_files["n_unresolved_ancestry_rna_files"].gt(0).sum()
        ) != 37):
    raise ValueError("Same-case additional RNA records differ from audited inputs")
print("Simple primary candidates:", len(rna_e_simple_files))
print("Multiplicity cases outside primary:", 1370 - len(simple_case_ids))
print("Same-case additional RNA:", 530, "| additional ambiguous ancestry:", 37)


## STAR Counts Technical Characterization

Read each uniquely represented primary STAR Counts file once. Record gene-row structure, version-normalized Ensembl ID duplication, `unstranded` count validity, assigned-gene count totals, detected genes and STAR special-category counts. These are source diagnostics, not sequencing-quality acceptance criteria.


In [ ]:
# =============================================================================
# Characterize audited primary STAR Counts source files
# =============================================================================
STAR_ROOT = ROOT / "data/raw/cptac3/rnaseq_star_counts"
SPECIAL = ("N_unmapped", "N_multimapping", "N_noFeature", "N_ambiguous")
rows = []
for n, record in enumerate(rna_e_simple_files.itertuples(index=False), 1):
    star_path = STAR_ROOT / str(record.file_id) / str(record.filename)
    if not star_path.is_file() or star_path.stat().st_size != int(record.size_bytes):
        raise FileNotFoundError(f"Missing or size-mismatched STAR source: {star_path}")
    counts = pd.read_csv(
        star_path, sep="\t", comment="#", usecols=["gene_id", "unstranded"],
        dtype={"gene_id": "string", "unstranded": "string"},
    )
    if counts["gene_id"].isna().any():
        raise ValueError(f"Missing gene identifier in {record.file_id}")
    special_mask = counts["gene_id"].str.startswith("N_", na=False)
    gene_rows = counts.loc[~special_mask]
    if gene_rows.empty or gene_rows["gene_id"].duplicated().any():
        raise ValueError(f"Empty/duplicated STAR gene identifiers in {record.file_id}")
    unstranded = pd.to_numeric(gene_rows["unstranded"], errors="coerce").to_numpy(
        dtype="float64", na_value=np.nan
    )
    valid_counts = bool(
        np.isfinite(unstranded).all()
        and (unstranded >= 0).all()
        and (unstranded == np.floor(unstranded)).all()
    )
    special_values = counts.loc[special_mask]
    flags = {}
    for special_id in SPECIAL:
        observed = pd.to_numeric(
            special_values.loc[special_values["gene_id"].eq(special_id), "unstranded"],
            errors="coerce",
        )
        flags["star_" + special_id.lower()] = (
            float(observed.iloc[0])
            if len(observed) == 1 and pd.notna(observed.iloc[0]) else np.nan
        )
    rows.append({
        "case_uuid": record.case_uuid, "file_id": record.file_id,
        "sample_uuid": record.sample_uuid, "aliquot_uuid": record.aliquot_uuid,
        "primary_site": record.primary_site, "disease_type": record.disease_type,
        "n_gene_rows": int(len(gene_rows)),
        "n_star_special_rows": int(special_mask.sum()),
        "n_duplicate_versionless_ids": int(
            gene_rows["gene_id"].str.replace(r"\.\d+$", "", regex=True)
            .value_counts(dropna=False).gt(1).sum()
        ),
        "unstranded_nonnegative_integer_vector": valid_counts,
        "n_invalid_unstranded_gene_values": int(
            (~np.isfinite(unstranded) | (unstranded < 0)
             | (np.isfinite(unstranded) & (unstranded != np.floor(unstranded)))).sum()
        ),
        "unstranded_gene_count_total": (
            float(unstranded.sum()) if valid_counts else np.nan
        ),
        "n_detected_star_genes": int((unstranded > 0).sum()) if valid_counts else pd.NA,
        "n_other_rna_files_in_case": record.n_other_rna_files_in_case,
        "n_unresolved_ancestry_rna_files": record.n_unresolved_ancestry_rna_files,
        **flags,
    })
    if n % 200 == 0 or n == len(rna_e_simple_files):
        print("STAR source files audited:", n, "/", len(rna_e_simple_files))

rna_e_file_diagnostics = pd.DataFrame(rows).sort_values(
    ["case_uuid", "file_id"], kind="stable"
).reset_index(drop=True)
if (len(rna_e_file_diagnostics) != 1082
        or rna_e_file_diagnostics["file_id"].duplicated().any()
        or not rna_e_file_diagnostics["unstranded_nonnegative_integer_vector"].all()
        or rna_e_file_diagnostics["n_invalid_unstranded_gene_values"].sum() != 0):
    raise ValueError("STAR source diagnostics violate the primary candidate scope")
print(
    "STAR rows per file:",
    sorted(rna_e_file_diagnostics.n_gene_rows.unique().tolist()),
)
display(rna_e_file_diagnostics[[
    "n_gene_rows", "unstranded_gene_count_total", "n_detected_star_genes",
]].describe().T)
print("No QC threshold and no file exclusion applied")


## Additional RNA Files and Source Context

Preserve additional RNA files in the same case, including ambiguous sample-ancestry records. Additional files do not become alternative primary samples or technical replicates by assumption. Report the low-count tail descriptively without exclusions.


In [ ]:
# =============================================================================
# Characterize additional same-case RNA files and ancestry ambiguity
# =============================================================================
rna_f_other_files = idx_rna.loc[
    idx_rna["case_uuid"].isin(rna_e_simple_files["case_uuid"])
    & ~idx_rna["file_id"].isin(rna_e_simple_files["file_id"])
].copy()
extra_counts = rna_f_other_files.groupby("case_uuid").agg(
    n_extra_rna_files=("file_id", "nunique"),
    n_extra_ambiguous_ancestry=("mapping_status",
        lambda s: int(s.eq("ambiguous_sample_ancestry").sum())),
)
rna_f_case_context = rna_e_simple_files[[
    "case_uuid", "file_id", "primary_site", "disease_type",
    "n_other_rna_files_in_case", "n_unresolved_ancestry_rna_files",
]].copy()
rna_f_case_context["n_extra_rna_files"] = (
    rna_f_case_context["case_uuid"].map(extra_counts["n_extra_rna_files"])
    .fillna(0).astype("int64")
)
rna_f_case_context["n_extra_ambiguous_ancestry"] = (
    rna_f_case_context["case_uuid"].map(extra_counts["n_extra_ambiguous_ancestry"])
    .fillna(0).astype("int64")
)
if (not rna_f_case_context["n_extra_rna_files"].equals(
        rna_f_case_context["n_other_rna_files_in_case"].astype("int64"))
        or not rna_f_case_context["n_extra_ambiguous_ancestry"].equals(
            rna_f_case_context["n_unresolved_ancestry_rna_files"].astype("int64"))
        or rna_f_case_context["n_extra_rna_files"].gt(0).sum() != 530
        or rna_f_case_context["n_extra_ambiguous_ancestry"].gt(0).sum() != 37):
    raise ValueError("F same-case sources differ from audited 111 case identities")
rna_f_case_context["source_file_status"] = np.select(
    [
        rna_f_case_context["n_extra_ambiguous_ancestry"].gt(0),
        rna_f_case_context["n_extra_rna_files"].gt(0),
    ],
    ["OTHER_FILE_INCLUDES_AMBIGUOUS_ANCESTRY",
     "OTHER_SOURCE_RNA_FILES_NO_AMBIGUOUS_ANCESTRY"],
    default="NO_OTHER_SOURCE_RNA_FILES",
)
rna_f_case_context["analytic_eligibility"] = "NOT_FROZEN"
rna_f_low_count_tail = rna_e_file_diagnostics.sort_values(
    ["unstranded_gene_count_total", "file_id"], kind="stable"
).head(20).reset_index(drop=True)
rna_f_low_count_tail.insert(0, "descriptive_rank", range(1, 21))
rna_f_low_count_tail["qc_decision"] = "NOT_ASSESSED"
rna_f_low_count_tail["sample_excluded"] = False
print("Other RNA files:", len(rna_f_other_files), "| cases:", len(extra_counts))
print("Bottom-20 files are descriptive, not QC exclusions")
display(rna_f_low_count_tail[[
    "descriptive_rank", "file_id", "primary_site", "unstranded_gene_count_total"
]])


In [ ]:
# =============================================================================
# Summarize same-case RNA files without promoting alternative specimens
# =============================================================================

# Preserve resolved and ambiguous same-case RNA records without selecting alternatives.
comparators = rna_e_simple_files[[
    "case_uuid", "sample_uuid", "aliquot_uuid",
]].rename(columns={
    "sample_uuid": "provisional_primary_sample_uuid",
    "aliquot_uuid": "provisional_primary_aliquot_uuid",
})
rna_f_other_files = rna_f_other_files.merge(
    comparators, on="case_uuid", validate="many_to_one"
)
resolved_other = (
    rna_f_other_files["mapping_status"]
    .eq("unique_aliquot_sample")
    .fillna(False)
)
sample_type_original = rna_f_other_files["sample_type"].astype("string")
sample_lower = sample_type_original.fillna("").str.casefold()
rna_f_other_files["other_rna_source_class"] = np.select(
    [
        rna_f_other_files["mapping_status"]
        .eq("ambiguous_sample_ancestry")
        .fillna(False),
        ~resolved_other,
        sample_type_original.eq("Primary Tumor").fillna(False),
        sample_lower.str.contains("normal", regex=False),
        sample_lower.str.contains("recurrent", regex=False),
        sample_lower.str.contains("metastatic", regex=False),
    ],
    [
        "ANCESTRY_AMBIGUOUS_NO_SAMPLE_ATTRIBUTION",
        "OTHER_MAPPING_UNRESOLVED",
        "RESOLVED_PRIMARY_TUMOR_REVIEW",
        "RESOLVED_NORMAL_TISSUE",
        "RESOLVED_RECURRENT_TUMOR",
        "RESOLVED_METASTATIC_TUMOR",
    ],
    default="RESOLVED_OTHER_OR_UNSPECIFIED_SAMPLE_TYPE",
)
rna_f_other_files["same_exact_sample_as_primary"] = (
    resolved_other & rna_f_other_files["sample_uuid"].eq(
        rna_f_other_files["provisional_primary_sample_uuid"]
    ).fillna(False)
)
rna_f_other_files["same_exact_aliquot_as_primary"] = (
    resolved_other & rna_f_other_files["aliquot_uuid"].eq(
        rna_f_other_files["provisional_primary_aliquot_uuid"]
    ).fillna(False)
)
rna_f_other_files["eligibility_decision"] = "NOT_ASSESSED"
display(rna_f_other_files.groupby("other_rna_source_class").agg(
    source_files=("file_id", "nunique"),
    distinct_cases=("case_uuid", "nunique"),
    same_sample_files=("same_exact_sample_as_primary", "sum"),
    same_aliquot_files=("same_exact_aliquot_as_primary", "sum"),
).reset_index().sort_values("source_files", ascending=False))
if rna_f_other_files.loc[
    rna_f_other_files["other_rna_source_class"].eq(
        "ANCESTRY_AMBIGUOUS_NO_SAMPLE_ATTRIBUTION"
    ), "same_exact_sample_as_primary"
].any():
    raise ValueError("F ambiguous ancestry was incorrectly promoted")


### Source-Context Attrition and Additional-File Evidence

Summarize the primary-file candidate universe and original source context using case-level denominators. No TCGA subtype inference or external cohort eligibility decision is made in Notebook 112.


In [ ]:
# =============================================================================
# Characterize primary source-case representation by tumor context
# =============================================================================
primary_rna_case_context = (
    primary_rna.groupby("case_uuid", dropna=False)
    .agg(
        primary_site_values=("primary_site",
                             lambda v: "|".join(sorted(set(v.dropna())))),
        disease_type_values=("disease_type",
                             lambda v: "|".join(sorted(set(v.dropna())))),
    ).reset_index()
)
primary_rna_case_context["single_primary_file"] = (
    primary_rna_case_context["case_uuid"].isin(simple_case_ids)
)
primary_rna_case_context = primary_rna_case_context.merge(
    rna_e_file_diagnostics[[
        "case_uuid", "unstranded_nonnegative_integer_vector",
        "n_other_rna_files_in_case", "n_unresolved_ancestry_rna_files",
    ]],
    on="case_uuid", how="left", validate="one_to_one"
)
primary_rna_case_context["simple_count_vector_diagnostic_complete"] = (
    primary_rna_case_context["single_primary_file"]
    & primary_rna_case_context["unstranded_nonnegative_integer_vector"]
      .fillna(False).astype(bool)
)
primary_rna_case_context["simple_case_other_rna_source_files"] = (
    primary_rna_case_context["single_primary_file"]
    & primary_rna_case_context["n_other_rna_files_in_case"].gt(0).fillna(False)
)
primary_rna_case_context["simple_case_ambiguous_rna_source_files"] = (
    primary_rna_case_context["single_primary_file"]
    & primary_rna_case_context["n_unresolved_ancestry_rna_files"].gt(0).fillna(False)
)
rna_e_source_attrition = (
    primary_rna_case_context.groupby(
        ["primary_site_values", "disease_type_values"], dropna=False
    ).agg(
        n_provisional_primary_cases=("case_uuid", "nunique"),
        n_single_primary_file_cases=("single_primary_file", "sum"),
        n_single_complete_count_vector=(
            "simple_count_vector_diagnostic_complete", "sum"
        ),
        n_single_with_other_source_rna_files=(
            "simple_case_other_rna_source_files", "sum"
        ),
        n_single_with_ambiguous_source_rna_files=(
            "simple_case_ambiguous_rna_source_files", "sum"
        ),
    ).reset_index()
)
rna_e_source_attrition["n_multiplicity_cases_unadjudicated"] = (
    rna_e_source_attrition["n_provisional_primary_cases"]
    - rna_e_source_attrition["n_single_primary_file_cases"]
)
rna_e_source_attrition["fraction_cases_structurally_simple"] = (
    rna_e_source_attrition["n_single_primary_file_cases"]
    / rna_e_source_attrition["n_provisional_primary_cases"]
)
rna_e_source_attrition = rna_e_source_attrition.sort_values(
    ["n_provisional_primary_cases", "primary_site_values"],
    ascending=[False, True], kind="stable"
).reset_index(drop=True)
if (
    len(rna_e_source_attrition) != 22
    or rna_e_source_attrition["n_provisional_primary_cases"].sum() != 1370
    or rna_e_source_attrition["n_single_primary_file_cases"].sum() != 1082
    or rna_e_source_attrition["n_multiplicity_cases_unadjudicated"].sum() != 288
):
    raise ValueError("Primary RNA source-case representation violates audited counts")
display(rna_e_source_attrition)
print("Phase 4 exact frozen-feature coverage is NOT calculated here")


In [ ]:
# =============================================================================
# Summarize additional-file source contexts
# =============================================================================

# Summarize same-case source records without assigning specimen eligibility.
rna_f_context_summary = (
    rna_f_other_files.groupby(
        "case_uuid", dropna=False
    ).agg(
        additional_rna_files=("file_id", "nunique"),
        additional_normal_files=(
            "other_rna_source_class",
            lambda v: int(v.eq("RESOLVED_NORMAL_TISSUE").sum())
        ),
        additional_provisional_primary_files=(
            "other_rna_source_class",
            lambda v: int(v.eq("RESOLVED_PRIMARY_TUMOR_REVIEW").sum())
        ),
        additional_unresolved_ancestry_files=(
            "mapping_status",
            lambda v: int(v.eq("ambiguous_sample_ancestry").sum())
        ),
    ).reset_index()
)
rna_f_context_summary = (
    rna_e_simple_files[["case_uuid", "primary_site", "disease_type"]]
    .merge(rna_f_context_summary, on="case_uuid",
           how="left", validate="one_to_one")
)
for col in [
    "additional_rna_files", "additional_normal_files",
    "additional_provisional_primary_files", "additional_unresolved_ancestry_files",
]:
    rna_f_context_summary[col] = rna_f_context_summary[col].fillna(0).astype("int64")
rna_f_context_summary["cases_with_other_rna"] = (
    rna_f_context_summary["additional_rna_files"].gt(0)
)
rna_f_context_summary["cases_with_ambiguous_rna"] = (
    rna_f_context_summary["additional_unresolved_ancestry_files"].gt(0)
)
rna_f_context_summary = (
    rna_f_context_summary.groupby(
        ["primary_site", "disease_type"], dropna=False
    ).agg(
        simple_primary_cases=("case_uuid", "nunique"),
        cases_with_other_rna=("cases_with_other_rna", "sum"),
        cases_with_ambiguous_rna=("cases_with_ambiguous_rna", "sum"),
        additional_rna_files=("additional_rna_files", "sum"),
        additional_normal_files=("additional_normal_files", "sum"),
        additional_provisional_primary_files=(
            "additional_provisional_primary_files", "sum"
        ),
        additional_unresolved_ancestry_files=(
            "additional_unresolved_ancestry_files", "sum"
        ),
    ).reset_index().sort_values(
        ["simple_primary_cases", "primary_site"],
        ascending=[False, True], kind="stable"
    ).reset_index(drop=True)
)
if (
    len(rna_f_context_summary) != 18
    or rna_f_context_summary["simple_primary_cases"].sum() != 1082
    or rna_f_context_summary["additional_rna_files"].sum() != 544
    or rna_f_context_summary["cases_with_ambiguous_rna"].sum() != 37
):
    raise ValueError("Same-case source-context diagnostics violate audited counts")
display(rna_f_context_summary)
print("All strata are unharmonized source labels, not a TCGA lineage crosswalk")


In [ ]:
# =============================================================================
# Summarize source-context STAR count distributions
# =============================================================================
rna_g_file_qc_context = rna_e_file_diagnostics.merge(
    rna_f_case_context[[
        "case_uuid", "source_file_status",
        "n_extra_rna_files", "n_extra_ambiguous_ancestry",
    ]], on="case_uuid", validate="one_to_one"
)
rna_g_source_context = rna_g_file_qc_context.groupby(
    ["primary_site", "disease_type"], dropna=False
).agg(
    simple_primary_cases=("case_uuid", "nunique"),
    assigned_gene_count_min=("unstranded_gene_count_total", "min"),
    assigned_gene_count_q1=("unstranded_gene_count_total",
                            lambda v: float(v.quantile(0.25))),
    assigned_gene_count_median=("unstranded_gene_count_total", "median"),
    assigned_gene_count_q3=("unstranded_gene_count_total",
                            lambda v: float(v.quantile(0.75))),
    assigned_gene_count_max=("unstranded_gene_count_total", "max"),
    detected_gene_count_min=("n_detected_star_genes", "min"),
    detected_gene_count_median=("n_detected_star_genes", "median"),
    detected_gene_count_max=("n_detected_star_genes", "max"),
    with_other_rna=("n_extra_rna_files", lambda v: int(v.gt(0).sum())),
    with_ambiguous_additional_rna=("n_extra_ambiguous_ancestry",
                                    lambda v: int(v.gt(0).sum())),
).reset_index()
if rna_g_source_context["simple_primary_cases"].sum() != 1082:
    raise ValueError("G source-context totals do not reconcile to E")
rna_g_file_qc_context["quantitative_sequencing_qc_status"] = "NOT_ASSESSED"
rna_g_file_qc_context["qc_cutoff_applied"] = False
rna_g_file_qc_context["analytic_eligibility"] = "NOT_FROZEN"

rna_g_low_tail_file_ids = set(rna_f_low_count_tail["file_id"])
rna_g_tail_by_context = (
    rna_g_file_qc_context.loc[
        rna_g_file_qc_context["file_id"].isin(rna_g_low_tail_file_ids)
    ].groupby(["primary_site", "disease_type"], dropna=False)
    .agg(n_in_descriptive_low_20=("file_id", "nunique")).reset_index()
)
rna_g_source_context = rna_g_source_context.merge(
    rna_g_tail_by_context, on=["primary_site", "disease_type"],
    how="left", validate="one_to_one"
)
rna_g_source_context["n_in_descriptive_low_20"] = (
    rna_g_source_context["n_in_descriptive_low_20"].fillna(0).astype("int64")
)
rna_g_source_context["external_independent_qc_status"] = (
    "NOT_IN_CURRENT_AUDITED_EVIDENCE"
)
rna_g_source_context["lineage_crosswalk_status"] = "NOT_FROZEN"
if rna_g_source_context["n_in_descriptive_low_20"].sum() != 20:
    raise ValueError("G source-context descriptive low-20 count does not reconcile")

display(rna_g_source_context[[
    "primary_site", "disease_type", "simple_primary_cases",
    "assigned_gene_count_min", "assigned_gene_count_median",
    "assigned_gene_count_max",
]])
print("Descriptive source counts != validated sequencing QC or lineage eligibility")


## Sequencing QC Evidence and Limitations

Distinguish STAR raw-count characteristics, provider-reported metadata and genuinely independent sequencing/library QC. Availability of numerical fields is not proof of validated definitions, denominators, measurement independence, or comparability between libraries. No QC thresholds are introduced.


In [ ]:
# =============================================================================
# Prespecify QC evidence classes; 'available' is NOT an eligibility PASS
# =============================================================================

RNA_G_QC_METRIC_SPECS = [
    (
        "GDC_FILE_IDENTITY", "identity", "Notebook111 file-index, MD5 audit",
        "UUID / bytes / MD5", "ALL_2968_RNA_SOURCE_FILES",
        "AUDITED_SOURCE_IDENTITY", "technical acquisition identity only",
    ),
    (
        "EXACT_FROZEN_GENE_IDS",
        "feature_schema",
        "Notebook 113 source-cohort feature compatibility",
        "2389 exact ID presence flags", "REQUIRES_PHASE4_REFERENCE",
        "ASSESS_IN_NOTEBOOK_113", "identifier presence != expression or RNA quality",
    ),
    (
        "UNSTRANDED_ASSIGNED_GENE_COUNTS", "count_diagnostic",
        "Notebook 112 STAR unstranded counts",
        "counts per file across gene rows", "1082_SIMPLE_PRIMARY_FILES",
        "AVAILABLE_DESCRIPTIVE_ONLY", "assigned gene counts != total sequenced reads",
    ),
    (
        "STAR_DETECTED_GENE_ROWS", "count_diagnostic",
        "Notebook 112 STAR unstranded source counts",
        "genes with positive counts per file", "1082_SIMPLE_PRIMARY_FILES",
        "AVAILABLE_DESCRIPTIVE_ONLY", "context- and library-dependent detection",
    ),
    (
        "STAR_SPECIAL_N_ROWS", "count_diagnostic",
        "Notebook 112 STAR N_ tallies",
        "raw special-category counts, unstranded", "1082_SIMPLE_PRIMARY_FILES",
        "AVAILABILITY_SCREEN_ONLY", "no assumed independent read denominator",
    ),
    (
        "ALIGNED_READS_UNIQUE_MAPPING", "sequencing_qc",
        "STAR alignment log / validated read-group QC (not acquired)",
        "defined reads / fragments and mapping percentage",
        "NOT_IN_AUDITED_HANDOFF", "NOT_ASSESSED",
        "mapping numerator and denominator must use same assay/library",
    ),
    (
        "DUPLICATION_AND_LIBRARY_COMPLEXITY", "sequencing_qc",
        "RNA-seq alignment/library QC reports (not acquired)",
        "duplication and library complexity", "NOT_IN_AUDITED_HANDOFF",
        "NOT_ASSESSED", "depends on protocol, input depth, read pairing",
    ),
    (
        "RNA_INTEGRITY_AND_LIBRARY_PREP", "biospecimen_qc",
        "Independent RNA integrity/library QC not in audited handoffs",
        "RIN, library protocol and objective QC", "NOT_IN_AUDITED_HANDOFF",
        "NOT_ASSESSED",
        "GDC specimen provenance does not assess independent RNA integrity; "
        "it does not establish RIN or independent library sequencing QC",
    ),
    (
        "GENE_BODY_COVERAGE_5P3P", "sequencing_qc",
        "BAM-based RSeQC or RNA-SeQC (not acquired)",
        "gene-body profile / coverage", "NOT_IN_AUDITED_HANDOFF",
        "NOT_ASSESSED", "requires compatible independent alignment data",
    ),
    (
        "RRNA_MT_CONTAMINATION", "sequencing_qc",
        "RNA-seq aligned read QC (not acquired)",
        "rRNA/mitochondrial and contamination fractions",
        "NOT_IN_AUDITED_HANDOFF", "NOT_ASSESSED",
        "requires annotation and denominator harmonization",
    ),
    (
        "LIBRARY_STRANDEDNESS_AND_BATCH", "metadata_qc",
        "original library preparation/read-group metadata",
        "assay and strand protocol, batch IDs", "NOT_IN_AUDITED_HANDOFF",
        "NOT_ASSESSED", "unstranded counts harmonized, batches not removed",
    ),
    (
        "TUMOR_PURITY_PROLIFERATION", "biological_confounder",
        "source pathology/purity and independent context evidence",
        "purity estimate and proliferation covariates", "NOT_IN_AUDITED_HANDOFF",
        "NOT_ASSESSED", "these are confounders, not automatic QC exclusions",
    ),
]
rna_g_metric_evidence = pd.DataFrame(
    RNA_G_QC_METRIC_SPECS,
    columns=[
        "metric_id", "evidence_domain", "existing_or_required_source",
        "definition_or_unit", "observed_scope", "evidence_status",
        "interpretation_caveat",
    ],
)
rna_g_metric_evidence["qc_threshold_frozen"] = False
rna_g_metric_evidence["sample_eligibility_conferred"] = False
if (
    rna_g_metric_evidence["metric_id"].duplicated().any()
    or rna_g_metric_evidence["qc_threshold_frozen"].any()
):
    raise ValueError("Invalid prospective QC catalog schema")
display(rna_g_metric_evidence[[
    "metric_id", "observed_scope", "evidence_status"
]])
print("Metrics may be described; none has an empirical exclusion threshold")


In [ ]:
# =============================================================================
# Preserve STAR N_ measurements as raw tallies, not fake mapping-rate QC
# =============================================================================

rna_g_n_records = []
for name in SPECIAL:
    source_column = "star_" + name.lower()
    if source_column not in rna_e_file_diagnostics.columns:
        raise KeyError(
            "STAR source diagnostic lacks special-category column: "
            + source_column
        )
    values = pd.to_numeric(
        rna_e_file_diagnostics[source_column], errors="coerce"
    )
    n_finite = int(np.isfinite(values).sum())
    n_invalid = int((values.notna() & (~np.isfinite(values) | values.lt(0))).sum())
    rna_g_n_records.append({
        "star_source_label": name,
        "cached_column": source_column,
        "n_file_rows": len(values),
        "n_finite": n_finite,
        "n_missing_or_nonfinite": len(values) - n_finite,
        "n_negative_or_nonfinite_observed": n_invalid,
        "minimum_finite_count": float(values[np.isfinite(values)].min())
        if n_finite else np.nan,
        "maximum_finite_count": float(values[np.isfinite(values)].max())
        if n_finite else np.nan,
        "rate_denominator_defined": False,
        "mapping_rate_estimated": False,
        "quality_pass_assigned": False,
    })
rna_g_star_n_availability = pd.DataFrame(rna_g_n_records)
if (
    len(rna_g_star_n_availability) != len(SPECIAL)
    or rna_g_star_n_availability["rate_denominator_defined"].any()
    or rna_g_star_n_availability["quality_pass_assigned"].any()
):
    raise ValueError("Invalid STAR N_ availability/mapping interpretation")

display(rna_g_star_n_availability[[
    "star_source_label", "n_finite", "n_missing_or_nonfinite",
    "minimum_finite_count", "maximum_finite_count",
]])
print("No mapping, duplicate, effective-depth, or unassigned fractions inferred")


In [ ]:
# =============================================================================
# Prespecified independent QC evidence requirements before cohort eligibility
# =============================================================================

RNA_G_PROSPECTIVE_GATES = [
    (
        "INDEPENDENT_RNA_QC_PROVENANCE",
        "Original provider sequencing/alignment report with exact file/aliquot join",
        "Required to interpret mapping, duplication, library integrity, contamination",
        "NEEDS_SOURCE_DISCOVERY_NOT_ACQUIRED",
    ),
    (
        "MEASUREMENT_DEFINITION_AND_DENOMINATOR",
        "Assay and library-level read/fragment denominators, workflow release",
        "Required before any numerical RNA QC rate is comparable",
        "NOT_FROZEN",
    ),
    (
        "QC_ACCEPTABILITY_CRITERIA",
        "Independent published or provider-documented technical criteria",
        "Predeclare before QC-driven file selection; no quantile tuning",
        "NOT_FROZEN",
    ),
    (
        "LOW_COUNT_TAIL_INTERPRETATION",
        "Independent QC for low-count files, with source-context caveats",
        "Diagnostic review only; not an exclusion list",
        "NOT_ASSESSED",
    ),
    (
        "RNA_NORMALIZATION_AND_BATCH",
        "Frozen TCGA recipe and source-compatible RNA processing specification",
        "Gene ID agreement does not establish portable scaling",
        "NOT_FROZEN",
    ),
    (
        "LINEAGE_HARMONIZATION_AND_ATTRITION",
        "Source-to-TCGA lineage crosswalk and sample-selection sensitivity",
        "Avoid naive pan-cancer pooling, report non-evaluable lineages",
        "NOT_FROZEN",
    ),
    (
        "MULTIPLE_PRIMARY_CANDIDATE_POLICY",
        "Predeclared main-analysis versus sensitivity treatment of 288 cases",
        "No arbitrary aliquot or sample UUID choice; case as biological unit",
        "NOT_ASSESSED_IN_112",
    ),
    (
        "TCGA_CPTAC_DONOR_INDEPENDENCE",
        "Documented cohort-origin overlap crosswalk or provenance restrictions",
        "Lexical UUID inequality alone is not donor independence",
        "COHORT_LEVEL_EXTERNALITY_ASSUMPTION_NOT_DONOR_CERTIFICATION",
    ),
    (
        "EXTERNAL_REPLICATION_ESTIMAND_AND_NULL",
        "Frozen per-lineage/aggregate estimand and matched negative controls",
        "Required before looking at external program scores",
        "NOT_FROZEN",
    ),
]
rna_g_prevalidation_requirements = pd.DataFrame(
    RNA_G_PROSPECTIVE_GATES,
    columns=["gate_id", "evidence_needed", "scientific_reason", "gate_status"]
)
rna_g_prevalidation_requirements["pass_authorized"] = False
rna_g_prevalidation_requirements["biology_authorized"] = False


In [ ]:
# =============================================================================
# Validate the prespecified technical evidence boundaries
# =============================================================================

if (
    rna_g_prevalidation_requirements["gate_id"].duplicated().any()
    or rna_g_prevalidation_requirements["pass_authorized"].any()
):
    raise ValueError("Prospective RNA QC/prevalidation gates were improperly set")
display(rna_g_prevalidation_requirements[["gate_id", "gate_status"]])
print("Source-QC evidence availability is not QC eligibility")
# =============================================================================
# No sequencing-QC or biological eligibility conferred
# =============================================================================
if (
    len(rna_g_metric_evidence) != 12
    or len(rna_g_star_n_availability) != len(SPECIAL)
    or len(rna_g_prevalidation_requirements) != 9
    or len(rna_e_file_diagnostics) != 1082
    or rna_g_metric_evidence["sample_eligibility_conferred"].any()
    or rna_g_prevalidation_requirements["pass_authorized"].any()
    or rna_g_prevalidation_requirements["biology_authorized"].any()
    or rna_g_file_qc_context["qc_cutoff_applied"].any()
    or rna_f_low_count_tail["sample_excluded"].any()
):
    raise ValueError("Source QC evidence was incorrectly promoted into eligibility")
print("Source QC evidence catalogue: descriptive only")
print("Frozen 2,389-gene compatibility is assessed by Notebook 113")
print("Sample QC thresholds/exclusions and biological scores: NONE")


## Deterministic GDC STAR Output Metadata Sample

Select twelve uniquely represented primary STAR files from four source contexts using prespecified count ranks and stable file-ID tie-breaking. Compare the reconstructed IDs to the versioned GDC STAR-output metadata snapshot. This targeted sample tests metadata provenance; it is neither random nor representative of cohort-wide QC.


In [ ]:
# =============================================================================
# Select the prespecified twelve-file GDC source-metadata sample
# =============================================================================
RNA_H_SOURCE_STRATA = [
    ("Uterus, NOS", "Adenomas and Adenocarcinomas", "TWO_LOW_ONE_CENTRAL"),
    ("Pancreas", "Ductal and Lobular Neoplasms", "TWO_LOW_ONE_CENTRAL"),
    ("Kidney", "Adenomas and Adenocarcinomas", "ONE_LOW_TWO_CENTRAL"),
    ("Bronchus and lung", "Adenomas and Adenocarcinomas", "ONE_LOW_TWO_CENTRAL"),
]
pilot_parts = []
for site, disease, strategy in RNA_H_SOURCE_STRATA:
    group = rna_g_file_qc_context.loc[
        rna_g_file_qc_context["primary_site"].eq(site)
        & rna_g_file_qc_context["disease_type"].eq(disease)
    ].sort_values(
        ["unstranded_gene_count_total", "file_id"], kind="stable"
    ).reset_index(drop=True)
    if len(group) < 5:
        raise ValueError("Insufficient source files in prespecified source context")
    center = len(group) // 2
    positions = (
        [0, 1, center] if strategy == "TWO_LOW_ONE_CENTRAL"
        else [0, center - 1, center]
    )
    selected = group.iloc[positions][[
        "case_uuid", "file_id", "sample_uuid", "aliquot_uuid",
        "primary_site", "disease_type", "unstranded_gene_count_total",
    ]].copy()
    selected["technical_availability_stratum"] = strategy
    selected["technical_pilot_rank_position"] = positions
    pilot_parts.append(selected)
rna_h_pilot = pd.concat(pilot_parts, ignore_index=True).sort_values(
    ["primary_site", "technical_pilot_rank_position", "file_id"],
    kind="stable",
).reset_index(drop=True)
if (len(rna_h_pilot) != 12
        or rna_h_pilot["file_id"].duplicated().any()
        or rna_h_pilot["case_uuid"].duplicated().any()):
    raise ValueError("Deterministic source-metadata selection changed")
RNA_H_FILE_IDS = sorted(rna_h_pilot["file_id"].tolist())
with STAR_GDC_SNAPSHOT_PATH.open(encoding="utf-8") as handle:
    star_snapshot = json.load(handle)
if (star_snapshot.get("role") != "PREFLIGHT_H_METADATA_SOURCE_FEASIBILITY_PILOT_ONLY"
        or not isinstance(star_snapshot.get("hits"), list)
        or len(star_snapshot["hits"]) != 12
        or sorted(
            star_snapshot["query"]["filters"]["content"]["value"]
        ) != RNA_H_FILE_IDS
        or sorted(x["file_id"] for x in star_snapshot["hits"]) != RNA_H_FILE_IDS
        or star_snapshot.get("notebook111_rna_metadata_sha256")
        != audit_111["source_fingerprints"]["rnaseq"]["metadata"]["sha256"]):
    raise ValueError("Source sample does not match the versioned GDC snapshot")
query_digest = hashlib.sha256(json.dumps(
    star_snapshot["query"], sort_keys=True, separators=(",", ":")
).encode("utf-8")).hexdigest()
if query_digest != star_snapshot["query_sha256"]:
    raise ValueError("GDC STAR-output source query fingerprint mismatch")
star_hits = star_snapshot["hits"]
print("GDC STAR-output source sample reconstructed:", len(RNA_H_FILE_IDS))
display(rna_h_pilot[["primary_site", "disease_type",
                     "technical_pilot_rank_position", "file_id"]])


### STAR Output Identity and Workflow Metadata

Reconcile the selected STAR file identities, direct aliquot associations and workflow-input relationships with the recorded GDC metadata. Report missing or non-unique links explicitly rather than resolving them from case membership alone.


In [ ]:
# =============================================================================
# Reconcile STAR-output file identities with GDC metadata
# =============================================================================
rna_h_pilot = rna_h_pilot.merge(
    rna_e_simple_files[[
        "file_id", "filename", "size_bytes", "provider_md5"
    ]], on="file_id", validate="one_to_one"
)
rna_h_hits_by_file = {
    hit.get("file_id") or hit.get("id"): hit for hit in star_hits
}

# =============================================================================
# Validate source-file identity and direct aliquot association
# =============================================================================

rna_h_identity_rows = []
rna_h_nested_nodes = {}
rna_h_input_links_rows = []

for row in rna_h_pilot.itertuples(index=False):
    hit = rna_h_hits_by_file[row.file_id]
    if not (
        hit.get("file_name") == row.filename
        and str(hit.get("file_size")) == str(row.size_bytes)
        and str(hit.get("md5sum", "")).lower()
        == str(row.provider_md5).lower()
    ):
        raise ValueError("STAR-output identity mismatch: " + row.file_id)
    direct = hit.get("associated_entities")
    if not isinstance(direct, list):
        direct_status = "UNAVAILABLE_DIRECT_FILE_ASSOCIATION"
    else:
        matches = [
            entity for entity in direct
            if entity.get("entity_type") == "aliquot"
            and entity.get("entity_id") == row.aliquot_uuid
            and entity.get("case_id") == row.case_uuid
        ]
        direct_status = (
            "EXACT_DIRECT_ALIQUOT_CASE_LINK"
            if len(matches) == len(direct) == 1
            else "DIRECT_LINK_MISMATCH_OR_MULTIPLE"
        )
    analysis = hit.get("analysis")
    if not isinstance(analysis, dict):
        analysis = {}
    inputs = analysis.get("input_files")
    if inputs is None:
        inputs = []
        input_relation_status = "NO_INPUT_RELATION_EXPOSED"
    elif isinstance(inputs, list):
        input_relation_status = (
            "WORKFLOW_INPUT_LINK_PRESENT" if inputs
            else "EMPTY_WORKFLOW_INPUT_LINK"
        )
    else:
        raise ValueError("Unexpected GDC analysis.input_files structure")
    metadata = analysis.get("metadata")
    metadata = metadata if isinstance(metadata, dict) else {}
    groups = metadata.get("read_groups")
    groups = groups if isinstance(groups, list) else []
    qcs = [
        qc for grp in groups if isinstance(grp, dict)
        for qc in (grp.get("read_group_qcs") or [])
        if isinstance(qc, dict)
    ]
    rna_h_nested_nodes[row.file_id] = {
        "output_file": [hit],
        "input_file": [x for x in inputs if isinstance(x, dict)],
        "read_group": [x for x in groups if isinstance(x, dict)],
        "read_group_qc": qcs,
    }
    rna_h_identity_rows.append({
        "case_uuid": row.case_uuid, "file_id": row.file_id,
        "sample_uuid": row.sample_uuid, "aliquot_uuid": row.aliquot_uuid,
        "primary_site": row.primary_site,
        "disease_type": row.disease_type,
        "source_file_identity": "EXACT_FILENAME_BYTES_MD5_MATCH",
        "direct_file_aliquot_link": direct_status,
        "workflow_analysis_present": bool(analysis),
        "workflow_analysis_id": analysis.get("analysis_id"),
        "workflow_type": analysis.get("workflow_type"),
        "workflow_version": analysis.get("workflow_version"),
        "workflow_input_relation_status": input_relation_status,
        "n_input_file_nodes": len(inputs),
        "n_read_groups": len(groups),
        "n_read_group_qcs": len(qcs),
        "qc_eligibility": "NOT_FROZEN",
    })
    for item in inputs:
        if not isinstance(item, dict):
            raise ValueError("Unexpected non-dict workflow input metadata")
        rna_h_input_links_rows.append({
            "output_star_file_id": row.file_id,
            "case_uuid": row.case_uuid,
            "output_exact_direct_aliquot_link": direct_status,
            "workflow_input_file_id": item.get("file_id") or item.get("id"),
            "workflow_input_data_type": item.get("data_type"),
            "workflow_input_access": item.get("access"),
            "input_file_metadata_only": True,
            "paired_same_aliquot_independently_verified": False,
            "sequencing_qc_validated": False,
        })


In [ ]:
# =============================================================================
# Assemble the sampled STAR source-file provenance table
# =============================================================================

rna_h_file_identity = pd.DataFrame(rna_h_identity_rows).sort_values(
    ["primary_site", "case_uuid", "file_id"], kind="stable"
).reset_index(drop=True)
rna_h_input_links = pd.DataFrame(
    rna_h_input_links_rows,
    columns=[
        "output_star_file_id", "case_uuid",
        "output_exact_direct_aliquot_link",
        "workflow_input_file_id", "workflow_input_data_type",
        "workflow_input_access", "input_file_metadata_only",
        "paired_same_aliquot_independently_verified",
        "sequencing_qc_validated",
    ],
)
if (
    len(rna_h_file_identity) != 12
    or rna_h_file_identity["file_id"].duplicated().any()
    or rna_h_file_identity["qc_eligibility"].ne("NOT_FROZEN").any()
    or rna_h_input_links["sequencing_qc_validated"].any()
):
    raise ValueError("H QC input/source identity boundary violated")
display(rna_h_file_identity.groupby(
    ["primary_site", "direct_file_aliquot_link",
     "workflow_input_relation_status"]
).agg(
    n_star_files=("file_id", "nunique"),
    n_linked_input_nodes=("n_input_file_nodes", "sum"),
    n_read_group_qcs=("n_read_group_qcs", "sum"),
).reset_index())
print("Workflow input file presence != independent aliquot-level QC acceptance")


In [ ]:
# =============================================================================
# Preserve exact source scope for all probed independent quality dimensions
# =============================================================================

RNA_H_QC_PROBES = {
    "input_file": [
        "total_reads", "proportion_reads_mapped",
        "proportion_reads_duplicated",
        "average_base_quality", "average_read_length",
        "average_insert_size",
    ],
    "read_group": [
        "rin", "read_length", "library_strand",
        "library_preparation_kit_name",
    ],
    "read_group_qc": [
        "total_sequences", "percent_gc_content",
        "basic_statistics", "sequence_duplication_levels",
    ],
}
RNA_H_PROBE_FIELD_COUNT = sum(
    len(v) for v in RNA_H_QC_PROBES.values()
)
def h_informative_value(value):
    if value is None:
        return False
    if isinstance(value, str):
        return value.strip().casefold() not in (
            "", "na", "n/a", "unknown", "not reported", "not available",
            "null", "none", "not applicable",
        )
    if isinstance(value, bool):
        return True
    if isinstance(value, (int, float)):
        return bool(np.isfinite(value))
    return isinstance(value, (dict, list)) and bool(value)


In [ ]:
# =============================================================================
# Classify availability of source-level GDC metadata fields
# =============================================================================

rna_h_qc_field_rows = []
for record in rna_h_file_identity.itertuples(index=False):
    nodes = rna_h_nested_nodes[record.file_id]
    for scope, fields in RNA_H_QC_PROBES.items():
        for field in fields:
            contents = [
                node[field] for node in nodes[scope]
                if field in node and h_informative_value(node[field])
            ]
            rna_h_qc_field_rows.append({
                "star_file_id": record.file_id,
                "case_uuid": record.case_uuid,
                "primary_site": record.primary_site,
                "source_scope": scope,
                "source_field": field,
                "n_nodes_in_scope": len(nodes[scope]),
                "n_nodes_with_informative_field": len(contents),
                "field_informative_for_star_file": len(contents) > 0,
                "has_exact_direct_star_aliquot_link": (
                    record.direct_file_aliquot_link
                    == "EXACT_DIRECT_ALIQUOT_CASE_LINK"
                ),
                "input_file_aliquot_qc_link_verified": False,
                "assay_comparable_quality_metric_confirmed": False,
                "field_reuse_for_sample_qc": "NOT_AUTHORIZED",
            })
rna_h_qc_field_evidence = pd.DataFrame(rna_h_qc_field_rows)
rna_h_qc_field_summary = (
    rna_h_qc_field_evidence.groupby(
        ["source_scope", "source_field"]
    ).agg(
        n_pilot_star_files=("star_file_id", "nunique"),
        n_files_with_informative_field=(
            "field_informative_for_star_file", "sum"
        ),
        n_files_with_exact_direct_star_link=(
            "has_exact_direct_star_aliquot_link", "sum"
        ),
        n_files_with_independently_verified_input_qc=(
            "input_file_aliquot_qc_link_verified", "sum"
        ),
    ).reset_index()
)
if (
    len(rna_h_qc_field_evidence) != (
        len(rna_h_file_identity) * RNA_H_PROBE_FIELD_COUNT
    )
    or rna_h_qc_field_evidence["field_reuse_for_sample_qc"].ne(
        "NOT_AUTHORIZED"
    ).any()
    or rna_h_qc_field_summary[
        "n_files_with_independently_verified_input_qc"].sum() != 0
):
    raise ValueError("H availability probe drifted into QC validation")

display(rna_h_qc_field_summary)
print("Provider QC fields are evidence AVAILABILITY probes only")
print("No independent read/library quality gate or threshold was approved")


In [ ]:
# =============================================================================
# Summarize sampled source metadata without setting QC eligibility
# =============================================================================

# Summarize the bounded source-metadata sample (no eligibility decision).
rna_h_summary = {
    "pilot_star_files": len(rna_h_file_identity),
    "pilot_unique_cases": int(rna_h_file_identity["case_uuid"].nunique()),
    "exact_direct_aliquot_link_files": int(
        rna_h_file_identity["direct_file_aliquot_link"].eq(
            "EXACT_DIRECT_ALIQUOT_CASE_LINK"
        ).sum()
    ),
    "files_with_workflow_input_records": int(
        rna_h_file_identity["n_input_file_nodes"].gt(0).sum()
    ),
    "files_with_read_group_qc_records": int(
        rna_h_file_identity["n_read_group_qcs"].gt(0).sum()
    ),
    "files_with_any_informative_field": int(
        rna_h_qc_field_evidence.groupby("star_file_id")[
            "field_informative_for_star_file"
        ].any().sum()
    ),
    "field_availability_rows": int(len(rna_h_qc_field_evidence)),
    "independent_qc_eligibility_validated": False,
    "selection_or_exclusion_performed": False,
    "cohort_frozen": False,
}
if (
    rna_h_summary["pilot_star_files"] != 12
    or rna_h_summary["pilot_unique_cases"] != 12
    or rna_h_summary["field_availability_rows"] != 168
    or rna_h_summary["independent_qc_eligibility_validated"]
):
    raise ValueError(
        "Provider-field availability violates the prespecified sample scope"
    )
display(pd.Series(rna_h_summary, name="source_pilot_scope").to_frame())


## Upstream Input-File Provenance and Metadata Agreement

Authenticate the versioned exact-ID GDC upstream-input snapshot against the STAR workflow-input UUIDs. Reconcile direct case/aliquot identity and compare six provider metadata fields exposed through two GDC projections. Both projections may contain the same stored values; agreement is not an independent quality measurement.

GDC snapshots are read-only source inputs; their absence or inconsistency is a STOP, not a trigger for hidden API requests or for consuming downstream notebook outputs.


In [ ]:
# =============================================================================
# Associate STAR outputs with their recorded upstream input files
# =============================================================================

upstream_links = []
for star in star_hits:
    output_id = star.get("file_id") or star.get("id")
    matches = idx_rna.loc[idx_rna["file_id"].eq(output_id)]
    if len(matches) != 1:
        raise ValueError(
            "Source STAR file is absent or duplicated in the 111 inventory"
        )
    row = matches.iloc[0]
    if (
        row["mapping_status"] != "unique_aliquot_sample"
        or row["sample_type"] != "Primary Tumor"
        or str(row["filename"]) != str(star.get("file_name"))
        or int(row["size_bytes"]) != int(star.get("file_size") or -1)
        or str(row["provider_md5"]).lower()
        != str(star.get("md5sum")).lower()
    ):
        raise ValueError("STAR source identity contradicts Notebook 111")
    direct = star.get("associated_entities")
    if not isinstance(direct, list) or len(direct) != 1:
        raise ValueError("H output STAR direct association unexpectedly incomplete")
    entity = direct[0]
    if not (
        entity.get("entity_type") == "aliquot"
        and entity.get("entity_id") == row["aliquot_uuid"]
        and entity.get("case_id") == row["case_uuid"]
    ):
        raise ValueError(
            "STAR direct case/aliquot association disagrees with Notebook 111"
        )

    analysis = star.get("analysis") or {}
    if not isinstance(analysis, dict):
        raise ValueError("STAR workflow analysis metadata has invalid structure")
    inputs = analysis.get("input_files")
    if not isinstance(inputs, list) or len(inputs) != 1:
        raise ValueError(
            "Source sample requires exactly one workflow input per STAR output"
        )
    item = inputs[0]
    if not isinstance(item, dict) or not item.get("file_id"):
        raise ValueError("Workflow input lacks an exact source file UUID")
    upstream_links.append({
        "star_file_id": output_id,
        "case_uuid": str(row["case_uuid"]),
        "sample_uuid": str(row["sample_uuid"]),
        "aliquot_uuid": str(row["aliquot_uuid"]),
        "primary_site": str(row["primary_site"]),
        "disease_type": str(row["disease_type"]),
        "workflow_analysis_id": analysis.get("analysis_id"),
        "workflow_type": analysis.get("workflow_type"),
        "input_file_id": item["file_id"],
        "input_embedded": item,
    })

if (
    len(upstream_links) != 12
    or len({x["case_uuid"] for x in upstream_links}) != 12
    or len({x["star_file_id"] for x in upstream_links}) != 12
):
    raise ValueError("Sampled source case/file scope changed")
INPUT_UUIDS_112 = sorted({x["input_file_id"] for x in upstream_links})
if not 1 <= len(INPUT_UUIDS_112) <= 12:
    raise ValueError("112 pilot exceeds permitted source input UUID count")
# Some API projections omit analysis_id despite retaining the input-file
# relationship. Preserve missing identity as a source caveat, not a guessed ID.
n_with_workflow_analysis_id = sum(
    bool(x.get("workflow_analysis_id")) for x in upstream_links
)

print("Sampled STAR output files:", len(upstream_links))
print("STAR workflow analysis IDs populated:", n_with_workflow_analysis_id, "/ 12")
print("Exact upstream GDC input UUIDs:", len(INPUT_UUIDS_112))
print(
    "Repeated inputs, if any, remain shared source nodes — "
    "not independent replicates"
)


In [ ]:
# =============================================================================
# Authenticate versioned upstream GDC input-file metadata
# =============================================================================

GDC_FILES_URL = "https://api.gdc.cancer.gov/files"
FILES_FIELDS_112 = [
    "file_id", "file_name", "file_size", "md5sum", "access",
    "data_category", "data_type", "experimental_strategy",
    "total_reads", "proportion_reads_mapped",
    "proportion_reads_duplicated", "average_base_quality",
    "average_read_length", "average_insert_size",
    "analysis.analysis_id", "analysis.workflow_type",
    "analysis.workflow_version",
]
FILES_QUERY_112 = {
    "filters": {
        "op": "in",
        "content": {"field": "file_id", "value": INPUT_UUIDS_112},
    },
    "fields": ",".join(FILES_FIELDS_112),
    "expand": "associated_entities,analysis",
    "format": "JSON",
    "size": len(INPUT_UUIDS_112),
}
QUERY_DIGEST_112 = hashlib.sha256(
    json.dumps(
        FILES_QUERY_112,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
).hexdigest()
SOURCE_HASH_111 = calculate_sha256(AUDIT_111_PATH)
SOURCE_HASH_STAR = calculate_sha256(STAR_GDC_SNAPSHOT_PATH)

if not SNAPSHOT_112_PATH.is_file():
    raise FileNotFoundError(
        "Required versioned upstream GDC source snapshot is absent: "
        + str(SNAPSHOT_112_PATH)
    )

with SNAPSHOT_112_PATH.open(encoding="utf-8") as handle:
    snapshot_112 = json.load(handle)

if (
    snapshot_112.get("query_sha256") != QUERY_DIGEST_112
    or snapshot_112.get("upstream_111_summary_sha256") != SOURCE_HASH_111
    or snapshot_112.get("upstream_H_snapshot_sha256") != SOURCE_HASH_STAR
    or snapshot_112.get("requested_input_file_ids") != INPUT_UUIDS_112
):
    raise ValueError("Versioned upstream GDC snapshot provenance mismatch")

direct_hits_112 = snapshot_112.get("hits")
if not isinstance(direct_hits_112, list):
    raise ValueError("Upstream GDC snapshot lacks file records")
returned_112 = {
    hit.get("file_id") or hit.get("id"): hit
    for hit in direct_hits_112
}
if (
    len(direct_hits_112) != len(INPUT_UUIDS_112)
    or len(returned_112) != len(direct_hits_112)
    or set(returned_112) != set(INPUT_UUIDS_112)
):
    raise ValueError("Upstream GDC snapshot file identities are inconsistent")

print("Versioned upstream GDC records:", len(returned_112))
print("Source snapshot SHA-256:", calculate_sha256(SNAPSHOT_112_PATH))
print("No live GDC requests or source read-file downloads")


In [ ]:
# =============================================================================
# Reconcile direct upstream input associations with STAR source aliquots
# =============================================================================

provenance_rows_112 = []
for rel in upstream_links:
    direct_input = returned_112.get(rel["input_file_id"])
    if direct_input is None:
        parent_status = "INPUT_RECORD_NOT_RETURNED_BY_EXACT_QUERY"
        records = None
        n_direct = 0
    else:
        records = direct_input.get("associated_entities")
        if not isinstance(records, list):
            parent_status = "INPUT_DIRECT_ASSOCIATIONS_NOT_EXPOSED"
            n_direct = 0
        else:
            n_direct = len(records)
            parents = [
                e for e in records if isinstance(e, dict)
                and e.get("entity_type") == "aliquot"
                and e.get("case_id") == rel["case_uuid"]
                and e.get("entity_id") == rel["aliquot_uuid"]
            ]
            if len(parents) == n_direct == 1:
                parent_status = "EXACT_INPUT_STAR_CASE_ALIQUOT_MATCH"
            else:
                parent_status = "MISMATCH_OR_NONUNIQUE_INPUT_ASSOCIATION"
    h_input = rel["input_embedded"]
    if direct_input is None:
        input_identity_consistency = "DIRECT_INPUT_FILE_RECORD_UNAVAILABLE"
    elif (
        h_input.get("file_name")
        and direct_input.get("file_name")
        and h_input["file_name"] != direct_input["file_name"]
    ):
        input_identity_consistency = "INPUT_FILENAME_CONTRADICTION"
    elif (
        h_input.get("access")
        and direct_input.get("access")
        and h_input["access"] != direct_input["access"]
    ):
        input_identity_consistency = "INPUT_ACCESS_STATUS_DIFFERENCE"
    else:
        input_identity_consistency = "NO_OBSERVED_INPUT_METADATA_CONTRADICTION"

    provenance_rows_112.append({
        "star_file_id": rel["star_file_id"],
        "case_uuid": rel["case_uuid"],
        "sample_uuid": rel["sample_uuid"],
        "aliquot_uuid": rel["aliquot_uuid"],
        "primary_site": rel["primary_site"],
        "disease_type": rel["disease_type"],
        "star_workflow_analysis_id": rel["workflow_analysis_id"],
        "star_workflow_type": rel["workflow_type"],
        "input_file_id": rel["input_file_id"],
        "input_direct_file_record_returned": direct_input is not None,
        "input_direct_case_aliquot_status": parent_status,
        "n_direct_entities_input": n_direct,
        "h_input_file_name": h_input.get("file_name"),
        "direct_input_file_name": (
            direct_input.get("file_name") if direct_input else None
        ),
        "h_input_data_type": h_input.get("data_type"),
        "direct_input_data_type": (
            direct_input.get("data_type") if direct_input else None
        ),
        "direct_input_access": (
            direct_input.get("access") if direct_input else None
        ),
        "input_file_metadata_consistency": input_identity_consistency,
        "direct_input_record_is_independent_quality_measurement": False,
        "assay_qc_denominators_validated": False,
        "analytic_sample_eligibility": "NOT_FROZEN",
    })


In [ ]:
# =============================================================================
# Assemble upstream-input source-provenance records
# =============================================================================

input_provenance_112 = pd.DataFrame(provenance_rows_112).sort_values(
    ["primary_site", "case_uuid", "star_file_id"], kind="stable"
).reset_index(drop=True)
if (
    len(input_provenance_112) != 12
    or input_provenance_112["star_file_id"].duplicated().any()
    or input_provenance_112["analytic_sample_eligibility"].ne(
        "NOT_FROZEN"
    ).any()
):
    raise ValueError("112 exact STAR-to-input identity scope changed")

display(input_provenance_112.groupby(
    ["primary_site", "input_direct_case_aliquot_status"],
    dropna=False
).agg(
    n_star_files=("star_file_id", "nunique")
).reset_index())
print("Case identity alone never licenses assigning upstream QC to STAR aliquot")


In [ ]:
# =============================================================================
# Compare provider field values across GDC metadata projections
# =============================================================================

def finite_numeric(value):
    if value is None or isinstance(value, (bool, list, dict)):
        return None
    try:
        number = float(value)
    except (TypeError, ValueError, OverflowError):
        return None
    return number if math.isfinite(number) else None


metric_rows_112 = []
for rel in upstream_links:
    live = returned_112.get(rel["input_file_id"])
    old = rel["input_embedded"]
    for name in QC_FIELDS:
        old_n = finite_numeric(old.get(name))
        live_n = finite_numeric(live.get(name)) if live else None
        if old_n is None:
            status = "WORKFLOW_FIELD_NOT_NUMERIC"
        elif live is None:
            status = "DIRECT_FILE_RECORD_UNAVAILABLE"
        elif live_n is None:
            status = "DIRECT_FILE_METRIC_NOT_NUMERIC_OR_NOT_RETURNED"
        elif math.isclose(old_n, live_n, rel_tol=1e-9, abs_tol=1e-9):
            status = "AGREES_ACROSS_PROVIDER_PROJECTIONS"
        else:
            status = "VALUE_DRIFT_BETWEEN_PROVIDER_PROJECTIONS"
        metric_rows_112.append({
            "star_file_id": rel["star_file_id"],
            "input_file_id": rel["input_file_id"],
            "case_uuid": rel["case_uuid"],
            "primary_site": rel["primary_site"],
            "metric_id": name,
            "workflow_embedded_value": old_n,
            "direct_input_file_value": live_n,
            "provider_projections_agreement": status,
            "independent_measurement_proven": False,
            "qc_rate_denominator_and_tool_version_verified": False,
            "same_library_quality_comparability_verified": False,
            "qc_pass_assigned": False,
        })

metric_comparison_112 = pd.DataFrame(metric_rows_112).sort_values(
    ["star_file_id", "metric_id"], kind="stable"
).reset_index(drop=True)
if (
    len(metric_comparison_112) != 12 * len(QC_FIELDS)
    or metric_comparison_112.duplicated(
        ["star_file_id", "metric_id"]
    ).any()
    or metric_comparison_112["qc_pass_assigned"].any()
):
    raise ValueError("112 metrics violated availability/non-QC-policy scope")

display(metric_comparison_112.groupby(
    ["metric_id", "provider_projections_agreement"]
).size().rename("n_star_files").reset_index())
print("Agreement is metadata consistency; never independent measurement")
print("No mapping-rate thresholds, QC PASS/FAIL labels or biological scores")


In [ ]:
# =============================================================================
# Summarize upstream source identity and metadata agreement
# =============================================================================

n_exact_input_links = int(
    input_provenance_112["input_direct_case_aliquot_status"].eq(
        "EXACT_INPUT_STAR_CASE_ALIQUOT_MATCH"
    ).sum()
)
n_same_provider_metrics = int(
    metric_comparison_112["provider_projections_agreement"].eq(
        "AGREES_ACROSS_PROVIDER_PROJECTIONS"
    ).sum()
)
n_direct_records = int(
    input_provenance_112["input_direct_file_record_returned"].sum()
)
if n_exact_input_links == 12 and n_same_provider_metrics == 72:
    source_feasibility_112 = "EXACT_SOURCE_LINK_AND_PROVIDER_METRICS_RECONCILED"
elif n_direct_records == 0:
    source_feasibility_112 = "NO_PUBLIC_DIRECT_INPUT_RECORDS_IN_BOUNDED_QUERY"
else:
    source_feasibility_112 = "PARTIAL_OR_DISCORDANT_SOURCE_LINKAGE"

# Matching the exact input aliquot resolves a source-provenance ambiguity, but
# GDC's aligned-read quality fields still require their own protocol and
# measurement-denominator documentation before use as QC eligibility cutoffs.
source_summary_112 = pd.DataFrame([
    {"metric": "star_output_files", "value": len(upstream_links)},
    {"metric": "distinct_upstream_input_file_ids", "value": len(INPUT_UUIDS_112)},
    {"metric": "GDC_input_direct_file_records", "value": n_direct_records},
    {"metric": "exact_direct_case_aliquot_input_links", "value": n_exact_input_links},
    {"metric": "provider_metric_values_agree", "value": n_same_provider_metrics},
    {"metric": "total_probe_metric_rows", "value": len(metric_comparison_112)},
    {"metric": "sequencing_qc_pass_conferred", "value": 0},
    {"metric": "RNA_samples_selected_or_excluded", "value": 0},
    {"metric": "QC_cutoffs_frozen", "value": 0},
    {"metric": "cohort_frozen", "value": 0},
])
if (
    source_summary_112.loc[
        source_summary_112["metric"].eq("star_output_files"),
        "value"
    ].iloc[0] != 12
    or input_provenance_112[
        "direct_input_record_is_independent_quality_measurement"
    ].any()
):
    raise ValueError("Source feasibility conflated with independent QC")

print("112 source-link feasibility:", source_feasibility_112)
print("Independent QC definition, denominator, assay/library version: NOT VERIFIED")
print("Automatically expand to other files: NO")
print("Regulatory/biological QC acceptance, cohort/lineage/score selection: NO")


In [ ]:
# =============================================================================
# Persist a single downstream-consumed STAR technical inventory
# =============================================================================
SCAN_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
rna_e_file_diagnostics.to_csv(SCAN_OUTPUT, index=False)
if len(pd.read_csv(SCAN_OUTPUT, dtype="string")) != 1082:
    raise ValueError("112 technical inventory persistence did not conserve case rows")
print("Reproducible 112 technical inventory:", project_relative_path(SCAN_OUTPUT))

# =============================================================================
# Persist non-frozen Phase 1 source-QC handoff with exact input/output hashes
# =============================================================================

# The 12-row, 72-row and 10-row descriptive tables are displayed,
# not persisted: no downstream notebook consumes these intermediate CSVs.
tables_112 = {
    "source_identities": input_provenance_112,
    "metric_comparison": metric_comparison_112,
    "source_summary": source_summary_112,
}
for key, table in tables_112.items():
    print("Source-only diagnostic:", key, "rows:", len(table))
    display(table.head(15))


In [ ]:
# =============================================================================
# Write the downstream technical inventory provenance manifest
# =============================================================================

handoff_112 = {
    "notebook": 112,
    "role": "PHASE1_RNA_SOURCE_TECHNICAL_AND_QC_METADATA_AUDIT",
    "status": "BOUNDED_SOURCE_FEASIBILITY_ONLY_NOT_QUALITY_QC_PASS",
    "source_feasibility": source_feasibility_112,
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "prior_111_summary": {
        "path": project_relative_path(AUDIT_111_PATH),
        "sha256": SOURCE_HASH_111,
    },
    "prior_111_file_index": {
        "path": project_relative_path(FILE_INDEX_111_PATH),
        "sha256": calculate_sha256(FILE_INDEX_111_PATH),
    },
    "star_output_source_snapshot": {
        "path": project_relative_path(STAR_GDC_SNAPSHOT_PATH),
        "sha256": SOURCE_HASH_STAR,
        "query_sha256": star_snapshot["query_sha256"],
    },
    "input_direct_record_snapshot": {
        "path": project_relative_path(SNAPSHOT_112_PATH),
        "sha256": calculate_sha256(SNAPSHOT_112_PATH),
        "query_sha256": QUERY_DIGEST_112,
        "captured_at_utc": snapshot_112.get("captured_at_utc"),
    },
    "observations": {
        "star_output_files": len(upstream_links),
        "unique_upstream_input_IDs_requested": len(INPUT_UUIDS_112),
        "upstream_input_direct_records_returned": n_direct_records,
        "exact_direct_case_and_aliquot_links": n_exact_input_links,
        "agreeing_provider_metric_field_pairs": n_same_provider_metrics,
        "metric_field_pairs_total": len(metric_comparison_112),
    },
    "unresolved_gates": [
        "provider_field_tool_version_and_denominator_definitions",
        "comparable_library_and_assay_measurement_scope",
        "full_CPTAC3_QC_coverage_and_missingness_not_assessed",
        "lineage_and_sample_specific_QC_sensitivity_not_frozen",
        "cross_cohort_donor_independence_not_assessed_in_112",
        "RNA_case_multiplicity_policy_not_assessed_in_112",
    ],
    "data_access_scope": "AUDITED_LOCAL_STAR_COUNTS_AND_VERSIONED_GDC_METADATA",
    "payload_downloads": 0,
    "large_STAR_scans": 1,
    "source_star_count_files_read": len(rna_e_file_diagnostics),
    "automatic_API_expansion": False,
    "QC_cutoffs_frozen": 0,
    "samples_selected_or_excluded": 0,
    "cohort_frozen": False,
    "program_scores_computed": False,
    "biological_replication_performed": False,
    "source_technical_audit_complete": True,
    "QC_eligibility_authorized": False,
    "outputs": {
        "technical_inventory": {
            "path": project_relative_path(SCAN_OUTPUT),
            "rows": len(rna_e_file_diagnostics),
            "sha256": calculate_sha256(SCAN_OUTPUT),
        },
    },
}
with OUTPUT_112_PATHS["handoff"].open("w", encoding="utf-8") as h:
    json.dump(handoff_112, h, indent=2, ensure_ascii=False)
    h.write("\n")
with OUTPUT_112_PATHS["handoff"].open("r", encoding="utf-8") as h:
    persisted = json.load(h)
if (
    persisted["observations"] != handoff_112["observations"]
    or persisted["QC_eligibility_authorized"]
    or persisted["cohort_frozen"]
):
    raise ValueError("112 handoff persistence/eligibility boundary failed")

print("Handoff:", project_relative_path(OUTPUT_112_PATHS["handoff"]))
print(
    "Versioned GDC source snapshots are read-only; "
    "inferential eligibility is not established"
)


## Results and Scientific Limitations

The audited primary RNA-seq source universe contains 1,082 uniquely represented primary cases. The local STAR scan characterizes these source files and produces the technical inventory required by Notebook 113; multiply represented primary cases remain outside this technical inventory. A targeted twelve-file GDC provenance audit assesses exact STAR-to-input case/aliquot associations and provider field agreement.

The technical inventory and provenance manifest are written to `data/interim/metadata/` and authenticated by recorded SHA-256 fingerprints. Snapshot inputs are version-controlled under `docs/evidence/phase1/notebook112/`.

**Scientific limitations:** independent read/library-level sequencing QC, field-denominator definitions, library/batch comparability, cancer-lineage mapping, individual TCGA–CPTAC donor independence and external program replication are not established here. No threshold is used to select or exclude a patient and no downstream inference is authorized.
